# Taller 1 — Procesamiento de datasets, tokenizadores y embeddings (semántica vectorial)

**Maestría Virtual en Ingeniería de Sistemas y Computación · PLN · ACT_7**

Notebook **único** con los tres puntos del taller. Pensado para **Google Colab / Kaggle con GPU**.

| Punto | Tema | Herramientas |
|---|---|---|
| **1** | Embeddings de palabras sobre `spanish_billion_words` | Word2Vec + FastText (Gensim) |
| **2** | Tokenización del dataset **BioBERT** (local) | BETO (`dccuchile/bert-base-spanish-wwm-cased`) |
| **3** | Embeddings de oraciones sobre PDFs | `pymupdf4llm` + LangChain + `sentence-transformers` |

> **Antes de correr:**
> - Runtime con **GPU** (Punto 3 lo agradece; el Punto 1 usa CPU).
> - Sube a tu Drive la carpeta de la actividad en `MyDrive/PLN/ACT_7/` con:
>   - `Biobert/` (que contiene `Biobert_json.py` y `data56/{train,valid,test}.json`)
>   - `pdf-coding/` (los PDFs)

## 0. Configuración del entorno

In [2]:
# 0.1  Instalación de librerías (una vez por sesión)
!pip install -q -U gensim nltk datasets transformers sentence-transformers
!pip install -q -U pymupdf4llm langchain langchain-community pypdf

Listo. Si Colab lo pide, reinicia el entorno tras instalar.


In [3]:
# 0.2  Montar Drive y definir rutas (mismo esquema que las otras actividades)
import os
from google.colab import drive
# drive.mount('/content/drive')
drive.mount('/content/drive', force_remount=True)

BASE_DIR   = '/content/drive/MyDrive/PLN/ACT_7'
BIOBERT_DIR = os.path.join(BASE_DIR, 'Biobert')      # contiene Biobert_json.py y data56/
PDF_DIR     = os.path.join(BASE_DIR, 'pdf-coding')   # PDFs para el punto 3
MODELS_DIR  = os.path.join(BASE_DIR, 'models')       # modelos de embeddings entrenados
os.makedirs(MODELS_DIR, exist_ok=True)

for p in [BASE_DIR, BIOBERT_DIR, PDF_DIR]:
    print(f'{p}  ->  {"OK" if os.path.exists(p) else "NO EXISTE (súbelo a Drive)"}')

Mounted at /content/drive
/content/drive/MyDrive/PLN/ACT_7  ->  OK
/content/drive/MyDrive/PLN/ACT_7/Biobert  ->  OK
/content/drive/MyDrive/PLN/ACT_7/pdf-coding  ->  OK


In [4]:
# 0.3  Semilla y dispositivo
import numpy as np, random
try:
    import torch
    device = 'cuda' if torch.cuda.is_available() else 'cpu'
except Exception:
    device = 'cpu'
random.seed(42); np.random.seed(42)
print('Dispositivo:', device)

Dispositivo: cpu


---
# Punto 1 — Embeddings de palabras (Word2Vec y FastText con Gensim)

Sobre `spanish_billion_words` entrenamos **Word2Vec** y **FastText** y comparamos.

## 1.1  Preprocesamiento del dataset

Cargamos el corpus y para cada sentencia: minúsculas, **quitar números**, **quitar
puntuación/símbolos**, **quitar stopwords en español** y **tokenizar**. El resultado es una
**lista de listas** de tokens limpios (formato de entrada de Gensim).

In [5]:
# 1.1  Carga del corpus y limpieza -> lista de listas de tokens
import re
from datasets import load_dataset
import nltk
nltk.download('stopwords')
from nltk.corpus import stopwords
STOP_ES = set(stopwords.words('spanish'))

# Corpus en español limpio (mismo que se usó en ACT_1). Streaming para no agotar RAM.
ds_stream = load_dataset('jhonparra18/spanish_billion_words_clean', split='train', streaming=True)

def limpiar(texto):
    texto = texto.lower()
    texto = re.sub(r'\d+', ' ', texto)               # eliminar números
    texto = re.sub(r'[^a-záéíóúñü\s]', ' ', texto)   # solo letras del español
    tokens = texto.split()
    return [t for t in tokens if t not in STOP_ES and len(t) > 2]

def construir_corpus(n_sentencias):
    '''Devuelve una lista de listas de tokens limpios con n_sentencias del corpus.'''
    corpus = []
    for i, ejemplo in enumerate(ds_stream):
        if i >= n_sentencias:
            break
        toks = limpiar(ejemplo['text'])
        if toks:
            corpus.append(toks)
    return corpus

# Ejemplo pequeño para verificar el preprocesamiento
demo = construir_corpus(5)
print('Primeras sentencias tokenizadas y limpias:')
for s in demo[:3]:
    print(' ', s[:15], '...')

[nltk_data] Downloading package stopwords to /root/nltk_data...
[nltk_data]   Package stopwords is already up-to-date!


Resolving data files:   0%|          | 0/18 [00:00<?, ?it/s]

Primeras sentencias tokenizadas y limpias:
  ['source', 'wikisource', 'librodot', 'com', 'sensibilidad', 'jane', 'austen', 'capitulo'] ...
  ['familia', 'dashwood', 'llevaba', 'largo', 'tiempo', 'afincada', 'sussex'] ...
  ['propiedad', 'buen', 'tamaño', 'centro', 'encontraba', 'residencia', 'norland', 'park', 'manera', 'tan', 'digna', 'vivido', 'muchas', 'generaciones', 'llegó'] ...


## 1.2  Entrenamiento de los modelos

Según el taller: **Word2Vec** y **FastText** con embeddings de **100, 200 y 300** y tamaños de
corpus de **500 000, 5 M y 10 M** de sentencias.

> ⚠️ **Tiempo/RAM:** 5 M y 10 M de sentencias son muy pesados. Empieza con `500_000` para validar
> y sube según tu Colab (Pro/High-RAM para 5–10 M). Cada modelo se guarda en Drive para no reentrenar.

In [ ]:
# 1.2  Entrenamiento parametrizable de Word2Vec y FastText
from gensim.models import Word2Vec, FastText
import time

def entrenar_embeddings(n_sentencias, dim, algoritmo='w2v', workers=4, epochs=5):
    corpus = construir_corpus(n_sentencias)
    t0 = time.time()
    Clase = Word2Vec if algoritmo == 'w2v' else FastText
    modelo = Clase(sentences=corpus, vector_size=dim, window=5, min_count=5,
                   workers=workers, sg=1, epochs=epochs)   # sg=1 -> skip-gram
    ruta = os.path.join(MODELS_DIR, f'{algoritmo}_dim{dim}_n{n_sentencias}.model')
    modelo.save(ruta)
    print(f'[{algoritmo}] dim={dim} n={n_sentencias} | vocab={len(modelo.wv):,} '
          f'| {time.time()-t0:.1f}s -> {os.path.basename(ruta)}')
    return modelo

# Configuración recomendada para empezar (ajusta N y las combinaciones que pide el taller)
N = 500_000
w2v_300 = entrenar_embeddings(N, 300, 'w2v')
ft_300  = entrenar_embeddings(N, 300, 'ft')

for dim in (100, 200, 300):
    for n in (500_000, 5_000_000, 10_000_000):
        entrenar_embeddings(n, dim, 'w2v')
        entrenar_embeddings(n, dim, 'ft')

[w2v] dim=300 n=500000 | vocab=61,968 | 364.3s -> w2v_dim300_n500000.model


## 1.3  Consulta de similitud semántica (top-10 en AMBOS modelos)

Función que, dada una palabra, devuelve las **10 palabras más similares** en Word2Vec **y** en
FastText, para poder compararlos.

In [ ]:
# 1.3  Top-10 palabras más similares en ambos modelos
import pandas as pd

def similares(palabra, modelo_w2v=w2v_300, modelo_ft=ft_300, topn=10):
    palabra = palabra.lower()
    def _top(m):
        if palabra in m.wv:
            return [(w, round(s, 4)) for w, s in m.wv.most_similar(palabra, topn=topn)]
        return [('(fuera de vocab)', 0.0)]
    w2v = _top(modelo_w2v)
    # FastText sí maneja palabras fuera de vocabulario (subword)
    ft = [(w, round(s, 4)) for w, s in modelo_ft.wv.most_similar(palabra, topn=topn)]
    tabla = pd.DataFrame({'Word2Vec': [f'{w} ({s})' for w, s in w2v],
                          'FastText': [f'{w} ({s})' for w, s in ft]})
    tabla.index = [f'#{i+1}' for i in range(len(tabla))]
    print(f'Palabras más similares a: "{palabra}"')
    return tabla

similares('rey')

**¿Cuál es el mejor modelo?** *FastText* suele ganar porque construye vectores a partir de
**subpalabras (n-gramas de caracteres)**, así que maneja palabras raras, morfología rica y
términos fuera de vocabulario (OOV) — muy útil en español. *Word2Vec* es más rápido y puede ser
suficiente si el vocabulario está bien cubierto. *(Sustituye por tu conclusión con ejemplos.)*

## 1.4  Visualización con PCA / t-SNE

Proyectamos los embeddings **de tamaño 300** al plano para **50, 100 y 300** sentencias y
comparamos Word2Vec vs. FastText.

In [ ]:
# 1.4  Visualización 2D (PCA) de los embeddings dim=300 para 50/100/300 sentencias
from sklearn.decomposition import PCA
import matplotlib.pyplot as plt

def visualizar(n_sent, max_palabras=60):
    # Reentrenamos rápido con pocas sentencias solo para la visualización comparativa
    m_w2v = entrenar_embeddings(n_sent, 300, 'w2v', epochs=5)
    m_ft  = entrenar_embeddings(n_sent, 300, 'ft',  epochs=5)

    fig, axes = plt.subplots(1, 2, figsize=(15, 6))
    for ax, (m, nombre) in zip(axes, [(m_w2v, 'Word2Vec'), (m_ft, 'FastText')]):
        palabras = list(m.wv.index_to_key)[:max_palabras]
        X = np.array([m.wv[w] for w in palabras])
        coords = PCA(n_components=2, random_state=42).fit_transform(X)
        ax.scatter(coords[:, 0], coords[:, 1], s=15, alpha=.6)
        for (x, y), w in zip(coords, palabras):
            ax.annotate(w, (x, y), fontsize=7, alpha=.8)
        ax.set_title(f'{nombre} — dim 300 — {n_sent} sentencias', fontweight='bold')
        ax.grid(alpha=.2)
    plt.tight_layout(); plt.show()

for n in (50, 100, 300):
    visualizar(n)

---
# Punto 2 — Tokenización con BETO para el dataset BioBERT

Dataset local anotado con `["sentencia", "tag"]` (NER biomédico en español, 30 etiquetas).

## 2.1  Cargar train / validation / test con el script `Biobert_json`

El script `Biobert_json.py` (incluido en la carpeta) define el *loader*. Lo cargamos con
`load_dataset` apuntando a esa ruta.

In [ ]:
# 2.1  Carga del dataset con el script local
from datasets import load_dataset

# Ruta al script del loader (dentro de Biobert/ está Biobert_json.py y data56/)
# La carpeta con el .py debe contener también data56/. Ajusta si tu estructura difiere.
script_dir = os.path.join(BIOBERT_DIR)  # p.ej. .../ACT_7/Biobert  (donde vive Biobert_json.py)
# Si el .py está un nivel más adentro (Biobert/Biobert/), usa ese:
if not os.path.exists(os.path.join(script_dir, 'Biobert_json.py')):
    script_dir = os.path.join(BIOBERT_DIR, 'Biobert')

ds_bio = load_dataset(os.path.join(script_dir, 'Biobert_json.py'),
                      name='Biobert_json', trust_remote_code=True)
print(ds_bio)

# Nombres de las etiquetas (ClassLabel del script)
tag_names = ds_bio['train'].features['tag'].feature.names
print('\n# etiquetas:', len(tag_names))
print(tag_names)

## 2.2  Mostrar `train[0]` y `validation[0]` con sus etiquetas

In [ ]:
# 2.2  Sentencias [0] con sus tags (id -> nombre)
def mostrar_ejemplo(split, i=0):
    ej = ds_bio[split][i]
    print(f'--- {split}[{i}] ---')
    print('Sentencia:', ej['sentencia'])
    print('Tags (id):', ej['tag'])
    print('Tags (nombre):', [tag_names[t] for t in ej['tag']])
    print('\nToken -> etiqueta:')
    for tok, t in zip(ej['sentencia'], ej['tag']):
        print(f'  {tok:20s} {tag_names[t]}')
    print()

mostrar_ejemplo('train', 0)
mostrar_ejemplo('validation', 0)

## 2.3  Tokenización de `train[0]` y `validation[0]` con el tokenizador de BETO

In [ ]:
# 2.3  Tokenizar con BETO (dccuchile/bert-base-spanish-wwm-cased)
from transformers import AutoTokenizer
tokenizer = AutoTokenizer.from_pretrained('dccuchile/bert-base-spanish-wwm-cased')

def tokenizar_beto(split, i=0):
    palabras = ds_bio[split][i]['sentencia']
    enc = tokenizer(palabras, is_split_into_words=True)          # respeta la pretokenización
    subtokens = tokenizer.convert_ids_to_tokens(enc['input_ids'])
    print(f'--- {split}[{i}] tokenizado con BETO ---')
    print('Palabras originales:', palabras)
    print('Subtokens BETO     :', subtokens)
    print()

tokenizar_beto('train', 0)
tokenizar_beto('validation', 0)

---
# Punto 3 — Embeddings de oraciones sobre PDFs

Leemos los PDFs de `pdf-coding`, los fragmentamos, generamos embeddings con **4 modelos de
sentence-transformers** y hacemos recuperación semántica por **similitud coseno**.

## 3.1  Carga y procesamiento de los PDFs (con `pymupdf4llm`)

In [ ]:
# 3.1  Leer todos los PDFs del directorio con pymupdf4llm (mejor preservación del layout)
import glob, pymupdf4llm

pdf_paths = sorted(glob.glob(os.path.join(PDF_DIR, '**', '*.pdf'), recursive=True))
print(f'PDFs encontrados: {len(pdf_paths)}')

documentos = []   # lista de (nombre_archivo, texto_markdown)
for ruta in pdf_paths:
    try:
        texto = pymupdf4llm.to_markdown(ruta)   # extrae a markdown
        documentos.append((os.path.basename(ruta), texto))
    except Exception as e:
        print('  Error en', os.path.basename(ruta), '->', e)
print('Documentos leídos:', len(documentos))
print('Ejemplo (primeros 300 car.):\n', documentos[0][1][:300] if documentos else '(vacío)')

## 3.2  Fragmentación (RecursiveCharacterTextSplitter, chunk_size=1000, overlap=200)

In [ ]:
# 3.2  Chunking con LangChain (parámetros de la rúbrica)
from langchain.text_splitter import RecursiveCharacterTextSplitter

text_splitter = RecursiveCharacterTextSplitter(
    chunk_size=1000,
    chunk_overlap=200,
    separators=["\n\n", "\n", ". ", " ", ""],
)

chunks = []       # texto de cada fragmento
chunks_meta = []  # archivo de origen de cada fragmento
for nombre, texto in documentos:
    for frag in text_splitter.split_text(texto):
        frag = frag.strip()
        if len(frag) > 30:            # descartar fragmentos triviales
            chunks.append(frag)
            chunks_meta.append(nombre)
print(f'Total de chunks: {len(chunks)}')
print('Ejemplo de chunk:\n', chunks[0][:300] if chunks else '(vacío)')

## 3.3  Generación de embeddings con 4 modelos de sentence-transformers

| Modelo | Dim | Uso |
|---|---|---|
| `all-mpnet-base-v2` | 768 (monolingüe) | Alta precisión |
| `all-MiniLM-L6-v2` | 384 (monolingüe) | Rápido y eficiente |
| `paraphrase-multilingual-MiniLM-L12-v2` | 384 (multilingüe) | Similitud multilingüe |
| `BAAI/bge-m3` | 1024 (multilingüe) | Excelente para español |

In [ ]:
# 3.3  Embeddings de todos los chunks con cada modelo
from sentence_transformers import SentenceTransformer

MODELOS_ST = [
    'sentence-transformers/all-mpnet-base-v2',
    'sentence-transformers/all-MiniLM-L6-v2',
    'sentence-transformers/paraphrase-multilingual-MiniLM-L12-v2',
    'BAAI/bge-m3',
]

emb_por_modelo = {}   # nombre_modelo -> (modelo, matriz_embeddings)
for nombre in MODELOS_ST:
    print('Cargando y codificando con:', nombre)
    modelo = SentenceTransformer(nombre, device=device)
    emb = modelo.encode(chunks, batch_size=32, convert_to_numpy=True,
                        normalize_embeddings=True, show_progress_bar=True)
    emb_por_modelo[nombre] = (modelo, emb)
    print('  ->', emb.shape)

## 3.4  Consulta de similitud semántica (fragmento más similar por modelo)

In [ ]:
# 3.4  Dada una oración de entrada, recuperar el fragmento más similar en cada modelo
from sklearn.metrics.pairwise import cosine_similarity

consulta = "¿Cuáles son los objetivos del plan estratégico de desarrollo?"  # edita tu consulta

resultados_top = {}   # nombre_modelo -> (idx_mejor, score)
for nombre, (modelo, emb) in emb_por_modelo.items():
    q = modelo.encode([consulta], convert_to_numpy=True, normalize_embeddings=True)
    sims = cosine_similarity(q, emb)[0]
    idx = int(np.argmax(sims))
    resultados_top[nombre] = (idx, float(sims[idx]))
    print(f'\n=== Modelo: {nombre} ===')
    print(f'Similitud coseno: {sims[idx]:.4f}  (archivo: {chunks_meta[idx]})')
    print('Fragmento recuperado:\n', chunks[idx][:400], '...')

## 3.5  Visualización 2D con PCA (consulta vs. fragmento más similar por modelo)

In [ ]:
# 3.5  PCA por modelo: consulta (rojo) + fragmento más similar (verde), sobre el espacio de chunks
fig, axes = plt.subplots(2, 2, figsize=(14, 11))
for ax, (nombre, (modelo, emb)) in zip(axes.ravel(), emb_por_modelo.items()):
    q = modelo.encode([consulta], convert_to_numpy=True, normalize_embeddings=True)
    todos = np.vstack([emb, q])                      # chunks + consulta
    coords = PCA(n_components=2, random_state=42).fit_transform(todos)
    chunk_xy, q_xy = coords[:-1], coords[-1]
    idx = resultados_top[nombre][0]

    ax.scatter(chunk_xy[:, 0], chunk_xy[:, 1], s=12, alpha=.35, color='gray', label='chunks')
    ax.scatter(*chunk_xy[idx], s=180, color='green', marker='*', label='más similar')
    ax.scatter(*q_xy, s=140, color='red', marker='X', label='consulta')
    ax.set_title(nombre.split('/')[-1], fontweight='bold', fontsize=10)
    ax.legend(fontsize=8); ax.grid(alpha=.2)

plt.suptitle(f'PCA — consulta vs. fragmento más similar\n"{consulta}"', fontweight='bold')
plt.tight_layout(); plt.show()

## Conclusiones

- **Punto 1:** FastText maneja mejor la morfología del español y palabras OOV gracias a los
  n-gramas de caracteres; Word2Vec es más ligero. La visualización PCA muestra cómo se agrupan
  las palabras según cada modelo al aumentar el número de sentencias.
- **Punto 2:** el dataset BioBERT se carga con su script; BETO tokeniza en **subpalabras**
  (`##`), por lo que una palabra puede generar varios subtokens (importante al alinear etiquetas
  NER).
- **Punto 3:** los modelos multilingües (`bge-m3`, `paraphrase-multilingual`) suelen recuperar
  mejor en español; `all-mpnet-base-v2` da alta precisión en inglés. La similitud coseno + PCA
  permiten ver qué fragmento recupera cada modelo para la misma consulta.